In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, desc, sum, stddev, mean
spark = SparkSession.builder.appName("WildChatDataProcessing").master("local[*]").config("spark.sql.shuffle.partitions", "16").config("spark.driver.memory", "8g").config("spark.sql.parquet.mergeSchema", "false").getOrCreate()

In [ ]:
#Pass the ingested data into a new DataFrame for exploration and cleansing

s2_workfile = spark.read.parquet("/Users/james/Projects/WildChat/data/WildChatData/ingested_wild_chat_data.parquet")
s2_workfile_c = 
s2_workfile.createOrReplaceGlobalTempView("wild_chat_data")

In [ ]:
#Spot check the data to make sure nothing went wrong during the write/transfer/read process

len(s2_workfile.columns)
s2_workfile.count()
s2_workfile.printSchema()
s2_workfile.select(s2_workfile.columns[:10]).show(5, truncate=False)

root
 |-- conversation_hash: string (nullable = true)
 |-- model: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- conversation: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- content: string (nullable = true)
 |    |    |-- country: string (nullable = true)
 |    |    |-- hashed_ip: string (nullable = true)
 |    |    |-- header: struct (nullable = true)
 |    |    |    |-- accept-language: string (nullable = true)
 |    |    |    |-- user-agent: string (nullable = true)
 |    |    |-- language: string (nullable = true)
 |    |    |-- redacted: boolean (nullable = true)
 |    |    |-- role: string (nullable = true)
 |    |    |-- state: string (nullable = true)
 |    |    |-- timestamp: timestamp (nullable = true)
 |    |    |-- toxic: boolean (nullable = true)
 |    |    |-- turn_identifier: long (nullable = true)
 |-- turn: long (nullable = true)
 |-- language: string (nullable = true)
 |-- openai_moderation: array (nu

+--------------------------------+------------------+-------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
#Duplication Check

s2_workfile.groupBy(s2_workfile.columns).count().filter("count > 1").show() # Exact row duplicates

dupes = s2_workfile.groupBy("conversation_hash").count().filter("count > 1") # Duplicate conversation hashes


In [ ]:
#Missing Values Check

s2_workfile.select([sum(col(c).isNull().cast("int")).alias(c)for c in s2_workfile.columns]).show()

In [ ]:
# Addressing Outliers

"""Simple threshold-based check"""
s2_workfile.filter(col("message_length") > 10000).show() # Example threshold for message length

""""Z-Score Method"""
stats = s2_workfile.select(mean(col("message_length")).alias("mean"), stddev(col("message_length")).alias("stddev")).collect()[0]
mean_length = stats["mean"]
stddev_length = stats["stddev"] 
s2_workfile_outliers = s2_workfile.withColumn("z_score", (col("message_length") - mean_length) / stddev_length).filter(col("z_score").abs() > 3) # Z-score threshold of 3
s2_workfile.withColumn("z_score", (col("message_length") - mean_length) / stddev_length).filter(col("z_score").abs() > 3).show() # Z-score threshold of 3

"""Interquartile Range (IQR) Method"""
quantiles = s2_workfile.approxQuantile("message_length", [0.25, 0.75], 0.01)
q1, q3 = quantiles
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
s2_workfile_iqr_outliers = s2_workfile.filter((col("message_length") < lower_bound) | (col("message_length") > upper_bound))

In [ ]:
#Filter Phase

